In [1]:
import json
!pip install shapely
!pip install pyproj
from shapely.geometry import shape
from shapely.ops import transform
from pyproj import Transformer
!pip install geopandas
import geopandas as gpd
import matplotlib.pyplot as plt
!pip install fiona
import fiona

### Some data analysis on what do we now have after cleaning it

In [2]:
# Define the GeoJSON file name
filename = 'newdelhibuildings_complete_zones_cleaned.geojson'

# Open and load the GeoJSON data
with open(filename, 'r') as f:
    data = json.load(f)

# Initialize sets for unique values of each property
unique_NAME = set()
unique_zone_category = set()
unique_ZONE_ASSIGNED = set()
unique_ASSIGNMENT_CONFIDENCE = set()

# Loop through each feature and add the property's value to the respective set
for feature in data['features']:
    properties = feature.get('properties', {})
    unique_NAME.add(properties.get('NAME'))
    unique_zone_category.add(properties.get('zone_category'))
    unique_ZONE_ASSIGNED.add(properties.get('ZONE_ASSIGNED'))
    unique_ASSIGNMENT_CONFIDENCE.add(properties.get('ASSIGNMENT_CONFIDENCE'))

# Print the unique values
print("Unique values for NAME:", unique_NAME)
print("Unique values for zone_category:", unique_zone_category)
print("Unique values for ZONE_ASSIGNED:", unique_ZONE_ASSIGNED)
print("Unique values for ASSIGNMENT_CONFIDENCE:", unique_ASSIGNMENT_CONFIDENCE)

Unique values for NAME: {'URBANISABLE AREA', 'SPORTS CENTRE', 'GOVERNMET OFFICE', 'STADIUM', 'PARLIAMENT HOUSE', 'COMMUNITY CENTRE', 'UNIVERSITY CENTRE', 'GOVERNMENT LAND', 'PARK', 'DISTRICT CENTRE', 'POLICE HEADQUATER', 'HOTEL', 'ELECTRICITY (POWER HOUSE SUB STATION)', 'SEWERAGE (TREATMENT PLANT)', 'CULTURAL COMPLEX', 'MANUFACTURING SERVICE AND REPAIR INDUSTRY', 'TRANSMISSION CENTRE', 'WATER BODIES', 'REGIONAL PARK', 'COMMUNITY PARK', 'RELIGIOUS', 'GENERAL BUSINESS', 'HOSPITAL', 'WAREHOUSING', 'WASTE LAND', 'RESIDENTIAL AREA', 'SPECIAL AREA', 'SOCIAL CULTURAL', 'AGRICULTURE', 'CITY PARK', 'SOLID WASTE (SANITERY LANDFILL)', 'COLD STORAGE', 'AIR CITY', 'POLICE', 'FOREIGN MISSION', 'PRESIDENT HOUSE', 'INDUSTRY', 'HISTORICAL MONUMENTS', 'WHOLE SALE', 'EDUCATION AND RESEARCH'}
Unique values for zone_category: {'COMMUNITY PARK', 'URBANISABLE AREA', 'SPORTS CENTRE', 'Other', 'RESIDENTIAL AREA', 'SPECIAL AREA', 'UNIVERSITY CENTRE', 'GOVERNMENT LAND', 'AGRICULTURE', 'MANUFACTURING SERVICE AND 

In [3]:
# Initialize sets for unique values of each property
unique_NAME = set()
unique_zone_category = set()
unique_ZONE_ASSIGNED = set()
unique_ASSIGNMENT_CONFIDENCE = set()

# Loop through each feature and add the property's value to the respective set
for feature in data['features']:
    properties = feature.get('properties', {})
    unique_NAME.add(properties.get('NAME'))
    unique_zone_category.add(properties.get('zone_category'))
    unique_ZONE_ASSIGNED.add(properties.get('ZONE_ASSIGNED'))
    unique_ASSIGNMENT_CONFIDENCE.add(properties.get('ASSIGNMENT_CONFIDENCE'))

# Print the unique values
print("Unique values for NAME:", unique_NAME)

Unique values for NAME: {'URBANISABLE AREA', 'SPORTS CENTRE', 'GOVERNMET OFFICE', 'STADIUM', 'PARLIAMENT HOUSE', 'COMMUNITY CENTRE', 'UNIVERSITY CENTRE', 'GOVERNMENT LAND', 'PARK', 'DISTRICT CENTRE', 'POLICE HEADQUATER', 'HOTEL', 'ELECTRICITY (POWER HOUSE SUB STATION)', 'SEWERAGE (TREATMENT PLANT)', 'CULTURAL COMPLEX', 'MANUFACTURING SERVICE AND REPAIR INDUSTRY', 'TRANSMISSION CENTRE', 'WATER BODIES', 'REGIONAL PARK', 'COMMUNITY PARK', 'RELIGIOUS', 'GENERAL BUSINESS', 'HOSPITAL', 'WAREHOUSING', 'WASTE LAND', 'RESIDENTIAL AREA', 'SPECIAL AREA', 'SOCIAL CULTURAL', 'AGRICULTURE', 'CITY PARK', 'SOLID WASTE (SANITERY LANDFILL)', 'COLD STORAGE', 'AIR CITY', 'POLICE', 'FOREIGN MISSION', 'PRESIDENT HOUSE', 'INDUSTRY', 'HISTORICAL MONUMENTS', 'WHOLE SALE', 'EDUCATION AND RESEARCH'}


### adding some geometric propertis 

In [4]:
# Set up a transformer to project from EPSG:4326 (CRS84) to EPSG:32643 (UTM Zone 43N for New Delhi)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32643", always_xy=True)

def project_geometry(geom):
    return transform(transformer.transform, geom)

# Load the cleaned GeoJSON file
filename = 'newdelhibuildings_complete_zones_cleaned.geojson'
with open(filename, 'r') as f:
    data = json.load(f)

# Calculate the area for each feature and add it as a new property.
# The area is computed in square meters.
for feature in data["features"]:
    # Convert the geometry into a Shapely geometry object
    geom = shape(feature["geometry"])
    # Reproject the geometry to a metric coordinate system
    projected_geom = project_geometry(geom)
    # Compute the area (in square meters)
    area = projected_geom.area
    # Add the area property to the feature's properties
    feature["properties"]["area"] = area

# Save the updated GeoJSON into a new file
with open('newdelhibuildings_complete_zones_cleaned_with_area.geojson', 'w') as f:
    json.dump(data, f, indent=4)

print("GeoJSON updated with area for each feature and saved as 'newdelhibuildings_complete_zones_cleaned_with_area.geojson'.")

GeoJSON updated with area for each feature and saved as 'newdelhibuildings_complete_zones_cleaned_with_area.geojson'.


In [5]:
import geopandas as gpd
import fiona
import matplotlib.pyplot as plt

filename = "newdelhibuildings_complete_zones_cleaned_with_area.geojson"

# Open the file using Fiona and then create a GeoDataFrame from its features
with fiona.open(filename, driver="GeoJSON") as src:
    gdf = gpd.GeoDataFrame.from_features(src, crs=src.crs)

# Display the first few rows
display(gdf.head())

# Summary statistics for the 'area' column (in square meters)
print("Area Summary Statistics:")
print(gdf["area"].describe())


,geometry,NAME,zone_category,ZONE_ASSIGNED,ASSIGNMENT_CONFIDENCE,area
0,"POLYGON ((77.19205 28.40691, 77.19219 28.40692...",REGIONAL PARK,Unknown,True,0.705607,188.723790
1,"POLYGON ((77.19251 28.40692, 77.19265 28.40694...",REGIONAL PARK,Unknown,True,0.705307,249.252494
2,"POLYGON ((77.19902 28.42580, 77.19915 28.42583...",REGIONAL PARK,Other,False,NaN,68.699458
3,"POLYGON ((77.19902 28.42816, 77.19908 28.42813...",REGIONAL PARK,Other,False,NaN,68.295717
4,"POLYGON ((77.19911 28.42830, 77.19920 28.42828...",REGIONAL PARK,Other,False,NaN,162.860292


Area Summary Statistics:
count    263427.000000
mean        232.693086
std        1066.327057
min           0.010419
25%          83.654883
50%         136.443918
75%         224.558601
max      324979.676562
Name: area, dtype: float64


In [ ]:

# Optional: Visualize features colored by area
gdf.plot(column="area", legend=True, figsize=(10,6))
plt.title("Features Colored by Area (sq. m)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


### creating orders

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point
from datetime import datetime, timedelta

# Helper function to generate a random point inside a polygon
def get_random_point_in_polygon(polygon):
    minx, miny, maxx, maxy = polygon.bounds
    while True:
        p = Point(np.random.uniform(minx, maxx), np.random.uniform(miny, maxy))
        if polygon.contains(p):
            return p

# Define the simulation month details: here April 2025 (30 days)
start_date = pd.Timestamp("2025-04-01")
num_days = 30  # April typically has 30 days

# Define simulation parameters for each area type ("NAME")
dist_params = {
    "WHOLE SALE": {
         "order_multiplier": 0.01,
         "price_mean": 90,
         "price_std": 15,
         "time_low": 9,
         "time_high": 17
    },
    "CITY PARK": {
         "order_multiplier": 0.002,
         "price_mean": 110,
         "price_std": 20,
         "time_low": 10,
         "time_high": 20
    },
    "HOTEL": {
         "order_multiplier": 0.008,
         "price_mean": 130,
         "price_std": 25,
         "time_low": 8,
         "time_high": 23
    },
    "FOREIGN MISSION": {
         "order_multiplier": 0.001,
         "price_mean": 150,
         "price_std": 30,
         "time_low": 10,
         "time_high": 18
    },
    "SOLID WASTE (SANITERY LANDFILL)": {
         "order_multiplier": 0.0005,
         "price_mean": 50,
         "price_std": 10,
         "time_low": 8,
         "time_high": 16
    },
    "CULTURAL COMPLEX": {
         "order_multiplier": 0.004,
         "price_mean": 120,
         "price_std": 20,
         "time_low": 11,
         "time_high": 22
    },
    "SEWERAGE (TREATMENT PLANT)": {
         "order_multiplier": 0.0003,
         "price_mean": 40,
         "price_std": 5,
         "time_low": 7,
         "time_high": 15
    },
    "SPORTS CENTRE": {
         "order_multiplier": 0.006,
         "price_mean": 100,
         "price_std": 18,
         "time_low": 9,
         "time_high": 21
    },
    "RELIGIOUS": {
         "order_multiplier": 0.005,
         "price_mean": 80,
         "price_std": 10,
         "time_low": 7,
         "time_high": 20
    },
    "MANUFACTURING SERVICE AND REPAIR INDUSTRY": {
         "order_multiplier": 0.007,
         "price_mean": 110,
         "price_std": 20,
         "time_low": 7,
         "time_high": 17
    },
    "COMMUNITY CENTRE": {
         "order_multiplier": 0.004,
         "price_mean": 100,
         "price_std": 15,
         "time_low": 8,
         "time_high": 20
    },
    "PRESIDENT HOUSE": {
         "order_multiplier": 0.0008,
         "price_mean": 200,
         "price_std": 30,
         "time_low": 10,
         "time_high": 18
    },
    "COLD STORAGE": {
         "order_multiplier": 0.005,
         "price_mean": 90,
         "price_std": 15,
         "time_low": 6,
         "time_high": 14
    },
    "EDUCATION AND RESEARCH": {
         "order_multiplier": 0.006,
         "price_mean": 85,
         "price_std": 10,
         "time_low": 8,
         "time_high": 16
    },
    "AGRICULTURE": {
         "order_multiplier": 0.007,
         "price_mean": 80,
         "price_std": 15,
         "time_low": 6,
         "time_high": 12
    },
    "URBANISABLE AREA": {
         "order_multiplier": 0.005,
         "price_mean": 110,
         "price_std": 20,
         "time_low": 8,
         "time_high": 18
    },
    "POLICE HEADQUATER": {
         "order_multiplier": 0.002,
         "price_mean": 120,
         "price_std": 20,
         "time_low": 6,
         "time_high": 14
    },
    "HOSPITAL": {
         "order_multiplier": 0.004,
         "price_mean": 100,
         "price_std": 20,
         "time_low": 8,
         "time_high": 20
    },
    "ELECTRICITY (POWER HOUSE SUB STATION)": {
         "order_multiplier": 0.001,
         "price_mean": 130,
         "price_std": 20,
         "time_low": 7,
         "time_high": 15
    },
    "RESIDENTIAL AREA": {
         "order_multiplier": 0.008,
         "price_mean": 95,
         "price_std": 15,
         "time_low": 7,
         "time_high": 23
    },
    "WASTE LAND": {
         "order_multiplier": 0.001,
         "price_mean": 60,
         "price_std": 10,
         "time_low": 8,
         "time_high": 16
    },
    "WAREHOUSING": {
         "order_multiplier": 0.005,
         "price_mean": 100,
         "price_std": 15,
         "time_low": 7,
         "time_high": 19
    },
    "HISTORICAL MONUMENTS": {
         "order_multiplier": 0.003,
         "price_mean": 150,
         "price_std": 25,
         "time_low": 9,
         "time_high": 17
    },
    "POLICE": {
         "order_multiplier": 0.002,
         "price_mean": 110,
         "price_std": 20,
         "time_low": 6,
         "time_high": 14
    },
    "PARK": {
         "order_multiplier": 0.004,
         "price_mean": 105,
         "price_std": 20,
         "time_low": 10,
         "time_high": 20
    },
    "SOCIAL CULTURAL": {
         "order_multiplier": 0.004,
         "price_mean": 115,
         "price_std": 20,
         "time_low": 10,
         "time_high": 22
    },
    "INDUSTRY": {
         "order_multiplier": 0.006,
         "price_mean": 105,
         "price_std": 15,
         "time_low": 7,
         "time_high": 17
    },
    "SPECIAL AREA": {
         "order_multiplier": 0.003,
         "price_mean": 100,
         "price_std": 20,
         "time_low": 8,
         "time_high": 18
    },
    "DISTRICT CENTRE": {
         "order_multiplier": 0.005,
         "price_mean": 110,
         "price_std": 20,
         "time_low": 8,
         "time_high": 20
    },
    "GOVERNMENT LAND": {
         "order_multiplier": 0.001,
         "price_mean": 120,
         "price_std": 25,
         "time_low": 9,
         "time_high": 17
    },
    "GENERAL BUSINESS": {
         "order_multiplier": 0.005,
         "price_mean": 100,
         "price_std": 20,
         "time_low": 8,
         "time_high": 20
    },
    "COMMUNITY PARK": {
         "order_multiplier": 0.004,
         "price_mean": 105,
         "price_std": 20,
         "time_low": 9,
         "time_high": 19
    },
    "TRANSMISSION CENTRE": {
         "order_multiplier": 0.001,
         "price_mean": 130,
         "price_std": 20,
         "time_low": 7,
         "time_high": 15
    },
    "GOVERNMET OFFICE": {  
         "order_multiplier": 0.002,
         "price_mean": 115,
         "price_std": 20,
         "time_low": 8,
         "time_high": 16
    },
    "PARLIAMENT HOUSE": {
         "order_multiplier": 0.001,
         "price_mean": 200,
         "price_std": 30,
         "time_low": 10,
         "time_high": 18
    },
    "UNIVERSITY CENTRE": {
         "order_multiplier": 0.006,
         "price_mean": 90,
         "price_std": 15,
         "time_low": 9,
         "time_high": 17
    },
    "REGIONAL PARK": {
         "order_multiplier": 0.003,
         "price_mean": 120,
         "price_std": 25,
         "time_low": 10,
         "time_high": 20
    },
    "STADIUM": {
         "order_multiplier": 0.005,
         "price_mean": 130,
         "price_std": 25,
         "time_low": 11,
         "time_high": 22
    },
    "AIR CITY": {
         "order_multiplier": 0.004,
         "price_mean": 140,
         "price_std": 30,
         "time_low": 8,
         "time_high": 20
    },
    "WATER BODIES": {
         "order_multiplier": 0.0008,
         "price_mean": 70,
         "price_std": 15,
         "time_low": 8,
         "time_high": 16
    }
}

# Default parameters for any area type not explicitly specified
default_params = {
    "order_multiplier": 0.005,
    "price_mean": 100,
    "price_std": 20,
    "time_low": 6,
    "time_high": 23
}

# List to accumulate simulated order records
orders = []

# Loop over each polygon/area in the GeoDataFrame
for idx, row in gdf.iterrows():
    area_type = row["NAME"]
    geom_area = row["area"]
    polygon = row["geometry"]
    
    # Get distribution parameters based on area type (fallback to default if not specified)
    params = dist_params.get(area_type, default_params)
    
    # Expected number of orders: geometric area multiplied by area-specific order multiplier
    expected_orders = geom_area * params["order_multiplier"]
    num_orders = max(1, np.random.poisson(lam=expected_orders))
    
    for _ in range(num_orders):
        price = max(10, np.random.normal(loc=params["price_mean"], scale=params["price_std"]))
        # Generate a random day in the month
        random_day = np.random.randint(0, num_days)
        # Generate a random hour (as a float) within the area-specific time bounds
        random_hour = np.random.uniform(params["time_low"], params["time_high"])
        # Combine date and time into a datetime using Timedelta addition
        order_datetime = start_date + pd.Timedelta(days=random_day) + pd.Timedelta(hours=random_hour)
        
        # Get a random point inside the polygon as the order location
        random_point = get_random_point_in_polygon(polygon)
        order_location = (random_point.x, random_point.y)
        
        orders.append({
            "polygon_idx": idx,
            "NAME": area_type,
            "geom_area": geom_area,
            "price": price,
            "order_datetime": order_datetime,
            "order_location": order_location
        })

# Convert simulated orders to a DataFrame
orders_df = pd.DataFrame(orders)

# Group and display summary statistics by area type
summary = orders_df.groupby("NAME").agg(
    total_orders=("price", "count"),
    avg_price=("price", "mean"),
    std_price=("price", "std"),
    avg_order_datetime=("order_datetime", lambda x: x.astype('int64').mean()),  # mean timestamp in ns
).reset_index()

# Convert the average order datetime from ns back to datetime
summary["avg_order_datetime"] = pd.to_datetime(summary["avg_order_datetime"])

print("Aggregated simulated orders by area type:")
print(summary)

# Visualize order time distribution (extracting hour of day) using boxplots by area type
orders_df['order_hour'] = orders_df['order_datetime'].dt.hour + orders_df['order_datetime'].dt.minute/60.0
plt.figure(figsize=(14,8))
orders_df.boxplot(column="order_hour", by="NAME", rot=90, grid=False)
plt.title("Order Hour Distribution by Area Type")
plt.suptitle("")
plt.xlabel("Area Type (NAME)")
plt.ylabel("Order Hour")
plt.tight_layout()
plt.show()

# Visualize overall order price distribution
plt.figure(figsize=(10,6))
plt.hist(orders_df["price"], bins=20, edgecolor='black')
plt.xlabel("Order Price")
plt.ylabel("Frequency")
plt.title("Overall Distribution of Order Prices")
plt.show()

# Display the first few simulated orders
print(orders_df.head())


In [ ]:
print(orders_df)

In [ ]:
!pip install folium
import folium
from folium.plugins import HeatMap

# Prepare the heatmap data.
# Note: order_location in orders_df is stored as (lon, lat); Folium expects [lat, lon]
heat_data = [[loc[1], loc[0]] for loc in orders_df["order_location"]]

# Optionally compute the mean location to center the map
mean_lat = orders_df["order_location"].apply(lambda x: x[1]).mean()
mean_lon = orders_df["order_location"].apply(lambda x: x[0]).mean()

# Create a base map centered on the mean location (example uses New Delhi)
base_map = folium.Map(location=[mean_lat, mean_lon], zoom_start=12)

# Add the heat map layer.
HeatMap(heat_data, radius=10, blur=15).add_to(base_map)

# Display the map (in Jupyter, simply output the map object)
base_map
